In [1]:
from oqd_compiler_infrastructure import Post, PrettyPrint
from oqd_core.analysis.analog.cfg import AnalogCFGBuilder
from oqd_core.analysis.analog.type_checker import AnalogTypeChecker
from oqd_core.frontend.analog import parse_analog
from oqd_core.analysis.analog.symbol_table import AnalogSymbolTableBuilder
from oqd_core.compiler.analog.passes.compile import compile_analog_circuit
from oqd_analog_emulator.interpreter import IRGenerator

printer = Post(PrettyPrint())

source = """
a=2 \n b = a + 3 \n H = %I %* %X \n c = true \n d = not c \n 
e = c and d \n f = c or d \n g = a <= b \n h = a >= b \n 
i = a == b \n k = a != b \n l = a - 1 \n m = a * b \n n = 2^3 \n
"""

source = """ 
a = 2 \n b = 5 \n 
if (a > 0) { \n b = 3 } \n
if (b < 0) { \n a = 5} \n
else { \n a = 10}
if (a < 0) { \n b = 10}
"""

source = """ 
n = 5 \n
while (n > 0) { \n n = n - 1}
"""

source = """
a = [2, 3, 4]
"""

source = """ 
r = qreg(2)
q0 = r[0]
q1 = r[1]
"""


source = """ 
a = sin(3.14 / 4)
b = abs(-5)
c = atan2(8, 5)
d = heaviside(2)
i = -4 + 4 * 1j
e = real(0 + 1j)
f = imag(1j)
g = conj(1 + 1j)
"""

source = """ 
r = qreg(2)
q0 = r[0]
q1 = r[1]
initialize(q0)
// initialize(q1)
// result = evolve(-(3.14 / 4) %* %X , 1, q0)
// result2 = evolve(-(3.14 / 4) %* %X , 1, q1)
result = evolve(-(3.14 / 4) %* (%X %* %X) , 1, r)
measurement = measure(q0)
"""

circuit = parse_analog(source)
cfg = AnalogCFGBuilder().run(circuit)
checker = AnalogTypeChecker(cfg)

symbol_analysis = AnalogSymbolTableBuilder(cfg, checker.dataflow_result)
symbol_table = symbol_analysis.symbol_table

circuit, cfg = compile_analog_circuit(circuit=circuit, cfg=cfg, symbol_table=symbol_table)


In [2]:
interpreter = IRGenerator(graph=cfg)
interpreter.run()
store = interpreter.status()
store
# store['q0'].state

[array([1.+0.j]), array([0.+0.j])]


{('r', 0): <oqd_analog_emulator.interpreter.QubitRegister at 0x1175ff5d0>,
 ('r', 1): <oqd_analog_emulator.interpreter.QubitRegister at 0x1175ff5d0>,
 'result': array([0.70738829+0.70682517j, 0.        +0.j        ]),
 'measurement': {0: {'00': 10}, 1: {'00': 10}}}

In [3]:
store[('r', 0)].state

Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.70738829+0.70682517j]
 [0.        +0.j        ]]

In [5]:
store[('r', 0)].time

1.0

In [6]:
cfg.to_dict()

{0: {'register_id': 0,
  'kind': 'start',
  'stmt': {},
  'preds': [],
  'succs': [1],
  'exit_nodes': [],
  'edge_labels': {}},
 1: {'register_id': 1,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'r',
   'value': {'class_': 'QuantumRegister', 'size': 2}},
  'preds': [0],
  'succs': [2],
  'exit_nodes': [],
  'edge_labels': {}},
 2: {'register_id': 2,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q0',
   'value': {'class_': 'Extract',
    'access': {'class_': 'Access', 'name': 'r'},
    'index': 0}},
  'preds': [1],
  'succs': [3],
  'exit_nodes': [],
  'edge_labels': {}},
 3: {'register_id': 3,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q1',
   'value': {'class_': 'Extract',
    'access': {'class_': 'Access', 'name': 'r'},
    'index': 1}},
  'preds': [2],
  'succs': [4],
  'exit_nodes': [],
  'edge_labels': {}},
 4: {'register_id': 4,
  'kind': 'stmt',
  'stmt': {'class_': 'Initialize',
   'targets': {'class_': 'Access', 